In [ ]:
import requests
import json
import time
from tqdm import tqdm
from bs4 import BeautifulSoup


def fetch_page_links(page_url):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    try:
        resp = requests.get(page_url, headers=headers, timeout=15)
        resp.raise_for_status()
    except Exception as e:
        print(f"    Ошибка загрузки: {e}")
        return []
    soup = BeautifulSoup(resp.text, 'html.parser')
    links = []
    for block in soup.find_all('div', class_='span4'):
        has_schematic = block.find('button', class_='format_schematic')
        has_non_free = block.find('button', class_='format_non_free')
        if has_schematic and not has_non_free:
            a = block.find('h3').find('a') if block.find('h3') else None
            if a and a.has_attr('href'):
                href = a['href']
                if not href.startswith('http'):
                    href = 'https://www.minecraft-schematics.com' + href
                links.append(href)
    return links


def main():
    total_pages = 1141
    base_list_url = 'https://www.minecraft-schematics.com/latest/'
    all_links = []
    output_file = 'all_free_schematics.json'

    pbar = tqdm(total=total_pages, desc='Сбор ссылок', unit='стр')
    for page in range(1, total_pages + 1):
        if page == 1:
            url = base_list_url
        else:
            url = f"{base_list_url}{page}/"

        page_links = fetch_page_links(url)
        all_links.extend(page_links)

        pbar.set_description(f'Стр.{page}: +{len(page_links)} ссылок (всего {len(all_links)})')
        pbar.update(1)

        # Промежуточное сохранение
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(all_links, f, indent=4, ensure_ascii=False)

        time.sleep(0.25)  # пауза для бережного отношения к серверу

    print(f"\n✅ Сбор завершён! Всего собрано {len(all_links)} ссылок.")
    print("Примеры ссылок (первые 5):")
    for link in all_links[:5]:
        print(link)

main()

Стр.818: +0 ссылок (всего 9555):  72%|███████▏  | 818/1141 [09:22<27:21,  5.08s/стр] 

    Ошибка загрузки: HTTPSConnectionPool(host='www.minecraft-schematics.com', port=443): Read timed out. (read timeout=15)


Стр.1141: +0 ссылок (всего 14586): 100%|██████████| 1141/1141 [12:52<00:00,  1.48стр/s] 


✅ Сбор завершён! Всего собрано 14586 ссылок.
Примеры ссылок (первые 5):
https://www.minecraft-schematics.com/schematic/29154/
https://www.minecraft-schematics.com/schematic/29153/
https://www.minecraft-schematics.com/schematic/29144/
https://www.minecraft-schematics.com/schematic/29137/
https://www.minecraft-schematics.com/schematic/29125/


In [2]:
import os
import time
import re
import json
import subprocess
import threading
import shutil
import undetected_chromedriver as uc
from tqdm import tqdm


def get_chrome_version():
    """Определяет major-версию Chrome."""
    paths = [
        r"C:\Program Files\Google\Chrome\Application\chrome.exe",
        r"C:\Program Files (x86)\Google\Chrome\Application\chrome.exe",
        os.path.expanduser(r"~\AppData\Local\Google\Chrome\Application\chrome.exe"),
    ]
    for path in paths:
        if os.path.exists(path):
            try:
                cmd = f'powershell -command "(Get-Item \'{path}\').VersionInfo.FileVersion"'
                result = subprocess.run(cmd, capture_output=True, text=True, shell=True)
                version = result.stdout.strip()
                if version:
                    return int(version.split(".")[0])
            except Exception:
                continue
    try:
        cmd = r'reg query "HKEY_CURRENT_USER\Software\Google\Chrome\BLBeacon" /v version'
        result = subprocess.run(cmd, capture_output=True, text=True, shell=True)
        match = re.search(r"(\d+)\.\d+\.\d+\.\d+", result.stdout)
        if match:
            return int(match.group(1))
    except Exception:
        pass
    return int(input("Введи major-версию Chrome (например 145): ").strip())


def create_driver(save_dir: str, chrome_version: int):
    """Создаёт Chrome с нужной версией ChromeDriver."""
    options = uc.ChromeOptions()
    prefs = {
        "download.default_directory": save_dir,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
    }
    options.add_experimental_option("prefs", prefs)
    return uc.Chrome(options=options, version_main=chrome_version)


def wait_for_download(save_dir: str, files_before: set, timeout: int = 120):
    """Ждёт появления нового файла в папке."""
    elapsed = 0
    while elapsed < timeout:
        time.sleep(0.2)
        elapsed += 0.2
        files_now = set(os.listdir(save_dir))
        new_files = files_now - files_before
        completed = [f for f in new_files
                     if not f.endswith((".crdownload", ".tmp", ".part"))]
        if completed:
            time.sleep(0.2)
            return completed[0]
    return None


def download_single(driver, schematic_url: str, save_dir: str, timeout: int = 120) -> bool:
    """Скачивает одну схематику."""
    try:
        if not schematic_url.endswith("/"):
            schematic_url += "/"

        # driver.get(schematic_url)
        # time.sleep(1)

        download_url = schematic_url + "download/action/?type=schematic"
        files_before = set(os.listdir(save_dir))
        driver.get(download_url)

        downloaded_file = wait_for_download(save_dir, files_before, timeout)

        if downloaded_file:
            filepath = os.path.join(save_dir, downloaded_file)
            file_size = os.path.getsize(filepath)

            # Проверяем что скачался не HTML
            if file_size < 5000:
                try:
                    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
                        start = f.read(200).strip().lower()
                        if start.startswith("<!doctype") or start.startswith("<html"):
                            os.remove(filepath)
                            return False
                except Exception:
                    pass
            return True
        return False
    except Exception:
        return False


def save_failed(failed_list: list, filepath: str = "failed.json"):
    """Сохраняет неудачные ссылки."""
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(failed_list, f, indent=2, ensure_ascii=False)


def split_urls(urls: list, n: int) -> list:
    """Делит список на n примерно равных частей."""
    k, m = divmod(len(urls), n)
    return [urls[i * k + min(i, m):(i + 1) * k + min(i + 1, m)] for i in range(n)]


def worker_func(driver, urls_chunk, save_dir, failed, success_count, lock, pbar):
    """Функция одного воркера — скачивает свою часть ссылок."""
    for url in urls_chunk:
        ok = download_single(driver, url, save_dir)

        with lock:
            pbar.update(1)
            if ok:
                success_count[0] += 1
            else:
                failed.append(url)
                save_failed(failed)
            pbar.set_description(f"✅ {success_count[0]} ❌ {len(failed)}")

        time.sleep(0.5)


def main(num_workers: int = None):
    # input_file = "all_free_schematics.json"
    input_file="failed_copy.json"
    if not os.path.exists(input_file):
        print(f"❌ Файл {input_file} не найден!")
        return

    with open(input_file, "r", encoding="utf-8") as f:
        urls = json.load(f)

    total = len(urls)

    if num_workers is None:
        num_workers = int(input(f"Количество воркеров (всего {total} ссылок): ").strip())
    num_workers = min(num_workers, total)

    # Общая папка и подпапки для каждого воркера
    base_dir = os.path.abspath(os.path.join(os.getcwd(), "../../data/web_data/"))
    os.makedirs(base_dir, exist_ok=True)

    worker_dirs = []
    for i in range(num_workers):
        d = os.path.abspath(os.path.join(base_dir, f"_worker_{i}"))
        os.makedirs(d, exist_ok=True)
        worker_dirs.append(d)

    # Определяем версию Chrome один раз
    chrome_version = get_chrome_version()

    # Запускаем все браузеры и открываем страницу логина
    drivers = []
    for i in range(num_workers):
        print(f"🚀 Запускаю Chrome #{i + 1}/{num_workers}...")
        driver = create_driver(worker_dirs[i], chrome_version)
        driver.get("https://www.minecraft-schematics.com/login/")
        drivers.append(driver)
        time.sleep(2)

    input(f"\n🔑 Залогинься во всех {num_workers} браузерах и нажми Enter... ")

    # Делим ссылки между воркерами
    chunks = split_urls(urls, num_workers)

    # Общее состояние
    failed = []
    success_count = [0]  # list чтобы можно было менять из потока
    lock = threading.Lock()
    pbar = tqdm(total=total, desc="✅ 0 ❌ 0", unit="файл")

    # Запускаем потоки
    threads = []
    for i in range(num_workers):
        t = threading.Thread(
            target=worker_func,
            args=(drivers[i], chunks[i], worker_dirs[i],
                  failed, success_count, lock, pbar),
            daemon=True,
        )
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    pbar.close()

    # Переносим все файлы в общую папку downloads/
    for d in worker_dirs:
        for filename in os.listdir(d):
            src = os.path.join(d, filename)
            dst = os.path.join(base_dir, filename)
            # Если файл с таким именем уже есть — добавляем суффикс
            if os.path.exists(dst):
                name, ext = os.path.splitext(filename)
                counter = 1
                while os.path.exists(dst):
                    dst = os.path.join(base_dir, f"{name}_{counter}{ext}")
                    counter += 1
            shutil.move(src, dst)
        # Удаляем пустую папку воркера
        try:
            os.rmdir(d)
        except OSError:
            pass

    # Итоги
    print(f"\n✅ Успешно: {success_count[0]}/{total}")
    print(f"❌ Неудачно: {len(failed)}/{total}")
    print(f"📁 Папка: {base_dir}")

    if failed:
        save_failed(failed)
        print("💾 Неудачные ссылки: failed.json")
    elif os.path.exists("failed.json"):
        os.remove("failed.json")

    # Закрываем браузеры
    for driver in drivers:
        try:
            driver.quit()
        except Exception:
            pass


if __name__ == "__main__":
    main(1)

🚀 Запускаю Chrome #1/1...


✅ 0 ❌ 10: 100%|██████████| 10/10 [21:26<00:00, 128.70s/файл]



✅ Успешно: 0/10
❌ Неудачно: 10/10
📁 Папка: c:\Users\ryabo\Documents\Диплом\DiffusionCraft\data\web_data
💾 Неудачные ссылки: failed.json
